# MRI Contrast — a hands-on walkthrough

This notebook teaches **why** MRI sequences produce the contrast they do, by
*simulating* them. Every image below is computed from the tissue properties
(T1, T2, proton density) of a brain phantom through the MR signal equations —
nothing is a photograph. Change a number, re-run a cell, and watch the contrast
change.

It drives the project's headless engine (`src/simulator.py`); no GUI needed.

**Contents**
1. The three tissue properties (T1, T2, PD)
2. T1-weighting · 3. T2-weighting · 4. Proton-density weighting
5. Inversion recovery — FLAIR (null CSF) and STIR (null fat)
6. All contrasts side by side
7. Going quantitative — a T1 map


In [ ]:
import os, sys
# Make the engine importable whether the notebook runs from examples/ or repo root.
for _cand in ("../src", "src", "../../src"):
    if os.path.isdir(_cand):
        sys.path.insert(0, os.path.abspath(_cand)); break

import numpy as np
import matplotlib.pyplot as plt

import tissue_db
from simulator import Simulator, default_params
from brainweb_loader import get_brainweb_or_synthetic
from phantom3d import get_slice

# Build a simulator viewing an axial mid-slice of a brain volume.
volume, source = get_brainweb_or_synthetic()
sim = Simulator()
sim.volume = volume
sim.native_fov = 220.0
sim.orientation = "axial"
sim.slice_idx = volume.shape[0] // 2
print(f"phantom: {source}  shape={volume.shape}  axial slice {sim.slice_idx}")

TISSUES = [(1, "CSF", "#4a9eff"), (2, "GM", "#69db7c"),
           (3, "WM", "#ff6b6b"), (4, "Fat", "#ffd43b")]

def show(over, title, ax=None):
    # Render default_params(**over) and display it; returns (image, metrics).
    img, m = sim.simulate(default_params(**over))
    if ax is None:
        _, ax = plt.subplots(figsize=(4.6, 4.6))
    ax.imshow(img, cmap="gray", origin="lower"); ax.set_title(title); ax.set_axis_off()
    return img, m

def tissue_signal(img):
    # Median signal in each tissue (1=CSF .. 4=Fat) of the current slice.
    labels = get_slice(sim.volume, sim.orientation, sim.slice_idx)
    return {name: round(float(np.median(img[labels == lab])), 3)
            for lab, name, _ in TISSUES if np.any(labels == lab)}

## 1. The three tissue properties

MR contrast comes from three tissue properties:

- **T1** — *longitudinal recovery* time. After excitation, magnetisation regrows
  toward equilibrium as `1 − e^(−TR/T1)`. Short-T1 tissue (fat, white matter)
  recovers fast → more signal at short **TR**.
- **T2** — *transverse decay* time. Signal decays as `e^(−TE/T2)`. Long-T2 tissue
  (CSF, fluid, edema) keeps signal at long **TE**.
- **PD** — *proton density*. The ceiling on how much signal a tissue can give.

A sequence is just a **window onto these curves**, chosen via TR, TE (and TI).


In [ ]:
tp = tissue_db.properties("3T")
print("Brain tissue properties at 3T")
print(f"{'tissue':6s}{'T1 (ms)':>9s}{'T2 (ms)':>9s}{'PD':>6s}")
for lab, name, _ in TISSUES:
    p = tp[lab]; print(f"{name:6s}{p['T1']:9.0f}{p['T2']:9.0f}{p['PD']:6.2f}")

fig, (axr, axd) = plt.subplots(1, 2, figsize=(12, 4))
TR = np.linspace(1, 6000, 300)
for lab, name, c in TISSUES[:3]:
    axr.plot(TR, 1 - np.exp(-TR / tp[lab]["T1"]), c, lw=2, label=f"{name} (T1={tp[lab]['T1']:.0f})")
axr.axvline(500, color="k", ls=":", lw=1); axr.text(560, 0.05, "short TR", fontsize=9)
axr.set(title="T1 recovery  (signal vs TR)", xlabel="TR (ms)", ylabel="recovered Mz / M0")
axr.legend(); axr.grid(alpha=0.3)
TE = np.linspace(0, 300, 300)
for lab, name, c in TISSUES[:3]:
    axd.plot(TE, np.exp(-TE / tp[lab]["T2"]), c, lw=2, label=f"{name} (T2={tp[lab]['T2']:.0f})")
axd.axvline(100, color="k", ls=":", lw=1); axd.text(108, 0.9, "long TE", fontsize=9)
axd.set(title="T2 decay  (signal vs TE)", xlabel="TE (ms)", ylabel="remaining Mxy")
axd.legend(); axd.grid(alpha=0.3)
fig.tight_layout()

## 2. T1-weighted contrast — short TR, short TE

Pick a **short TR** (~500 ms) so tissues separate on the *recovery* curve, and a
**short TE** (~15 ms) so T2 decay barely acts. Fast-recovering tissue is bright:

**Fat > White matter > Gray matter > CSF.**

This is the workhorse for anatomy — and the base for post-contrast imaging, since
gadolinium shortens T1 and makes enhancing tissue bright.


In [ ]:
img, m = show(dict(sequence="Spin Echo", TR=500, TE=15), "T1-weighted SE  (TR=500, TE=15)")
print("median signal:", tissue_signal(img))
print(f"scan time {int(m['scan_time']//60)}:{int(m['scan_time']%60):02d}, SNR(WM)={m['snr_wm']:.1f}")

## 3. T2-weighted contrast — long TR, long TE

Now flip it: **long TR** (~4000 ms) lets everything recover (removes T1
differences), and **long TE** (~100 ms) lets the *decay* curve separate tissues.
Long-T2 tissue stays bright:

**CSF > Gray matter > White matter.**

This is the workhorse for pathology — most lesions, edema and tumours have
elevated water content (long T2) and light up.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.2, 4.6))
show(dict(sequence="Spin Echo", TR=500, TE=15), "T1w (TR=500, TE=15)", axes[0])
img, _ = show(dict(sequence="Spin Echo", TR=4000, TE=100), "T2w (TR=4000, TE=100)", axes[1])
fig.tight_layout()
print("T2w median signal:", tissue_signal(img))

## 4. Proton-density weighting — long TR, short TE

Use a **long TR and a short TE** to suppress *both* T1 and T2 effects: what's left
is contrast from proton density. Gray matter (PD 0.80) edges out white matter
(0.65).

> Note: CSF is **not** the brightest here. Its T1 is ~4500 ms, so even at TR=3000 ms
> it hasn't fully recovered — true PD weighting of CSF would need a much longer TR.
> The simulator reproduces this real-world subtlety.


In [ ]:
img, _ = show(dict(sequence="Spin Echo", TR=3000, TE=15), "Proton-density (TR=3000, TE=15)")
print("median signal:", tissue_signal(img))

## 5. Inversion recovery — nulling a tissue

Add a 180° **inversion** pulse, then wait a time **TI** before reading out. The
longitudinal magnetisation starts at −M0 and recovers through **zero**. A tissue
read out exactly when its magnetisation crosses zero gives **no signal** — it is
*nulled*. The crossing happens near `TI ≈ T1 · ln 2`.

- **FLAIR** picks TI to null **CSF** (~2548 ms at 3T) → suppresses bright CSF so
  periventricular lesions stand out.
- **STIR** picks a short TI to null **fat** (~265 ms at 3T) → suppresses fat so
  edema/marrow pathology stands out.


In [ ]:
TR_flair = 9000.0
TI = np.linspace(0, 5000, 400)
fig, ax = plt.subplots(figsize=(9, 4))
for lab, name, c in TISSUES:
    T1 = tp[lab]["T1"]
    Mz = 1 - 2 * np.exp(-TI / T1) + np.exp(-TR_flair / T1)
    ax.plot(TI, Mz, c, lw=2, label=f"{name} (T1={T1:.0f})")
ax.axhline(0, color="gray", lw=1)
ax.axvline(2548, color="k", ls="--", lw=1.5, label="FLAIR TI=2548 (null CSF)")
ax.axvline(265, color="0.5", ls="--", lw=1.5, label="STIR TI=265 (null fat)")
ax.set(title="Inversion recovery: signed Mz vs TI (TR=9000)", xlabel="TI (ms)", ylabel="Mz / M0")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
# Where each curve crosses zero is exactly where that tissue is nulled.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.2, 4.6))
img_fl, _ = show(dict(sequence="Inversion Recovery", TR=9000, TE=90, TI=2548),
                 "FLAIR — CSF nulled", axes[0])
img_st, _ = show(dict(sequence="Inversion Recovery", TR=5000, TE=30, TI=265),
                 "STIR — fat nulled", axes[1])
fig.tight_layout()
print("FLAIR:", tissue_signal(img_fl), " (CSF ~ 0)")
print("STIR :", tissue_signal(img_st), " (Fat ~ 0)")

## 6. All contrasts, side by side

Same anatomy, same slice — five different windows onto the underlying physics.
Notice how CSF goes dark → bright → suppressed as we move T1w → T2w → FLAIR.


In [ ]:
protocols = [
    ("T1w SE",  dict(sequence="Spin Echo", TR=500, TE=15)),
    ("PD SE",   dict(sequence="Spin Echo", TR=3000, TE=15)),
    ("T2w SE",  dict(sequence="Spin Echo", TR=4000, TE=100)),
    ("FLAIR",   dict(sequence="Inversion Recovery", TR=9000, TE=90, TI=2548)),
    ("STIR",    dict(sequence="Inversion Recovery", TR=5000, TE=30, TI=265)),
    ("MPRAGE",  dict(sequence="Inversion Recovery", TR=2500, TE=3, TI=900, flip_angle=8)),
]
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (name, over) in zip(axes.ravel(), protocols):
    show(over, name, ax)
fig.tight_layout()

## 7. Going quantitative — a T1 map

Weighted images are *qualitative* — pixel values are arbitrary. **Quantitative
MRI** instead measures a physical property per voxel. Here a variable-flip-angle
acquisition is fit to a **T1 map in milliseconds**. Bulk white and gray matter
recover their textbook T1 (≈ 830 and ≈ 1330 ms); thin CSF reads a little *below*
its 4500 ms pure value because partial-volume blending at the ventricle margins
pulls it down — the same bias real quantitative MRI shows in small structures.
This is the bridge from teaching to research use.


In [ ]:
t1, _ = sim.simulate(default_params(sequence="Quantitative (qMRI)", qmri_display="T1 Map (VFA)"))
fig, ax = plt.subplots(figsize=(6.4, 5), constrained_layout=True)
im = ax.imshow(t1, cmap="viridis", origin="lower", vmin=0, vmax=4500)
ax.set_title("Quantitative T1 map (VFA)"); ax.set_axis_off()
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04); cb.set_label("T1 (ms)")
labels = get_slice(sim.volume, sim.orientation, sim.slice_idx)
for lab, name in [(3, "WM"), (2, "GM"), (1, "CSF")]:
    print(f"recovered {name} T1 = {np.median(t1[labels == lab]):.0f} ms")

---
### Where to go next
- Every image came from `Simulator.simulate(default_params(**overrides))` — try
  your own TR/TE/TI/flip-angle combinations in the cells above.
- Other sequences: `"Gradient Echo"`, `"FSE / TSE"`, `"Diffusion (DWI)"`,
  `"MR Angiography"`, `"fMRI (BOLD)"`, `"Echo Planar (EPI)"`.
- For the interactive version with sliders and live k-space, run `python src/app_qt.py`.
